# Malinois MPRA 6: more TFs — every JASPAR motif with HepG2 or K562 ChIP-seq

The 251 TFs with a function CAV were only those with a motif hit in the Kircher MPRA regions. ENCODE has TF ChIP-seq
for many more targets in HepG2 and K562, so many more JASPAR motifs can get a CAV, and many more Malinois variants
(and more motif occurrences per variant) can get a ΔP. Question: **with more TFs, does ΣΔP predict Malinois better,
and does its gain over (summed) motifbreakR grow?**

Everything is the same procedure as before, nothing tuned on the MPRA:
- **sites**: `Pooled Regions 1` (ChIP bound vs DNase-unbound, matched on PWM / GC / CpG, ≤ 2000 per class, no window
  overlapping a Kircher region);
- **fit**: `Pooled Regions 1c` (NT-v2 500M, layer 24, L2 logistic, 5-fold out-of-fold AUC);
- **Malinois hits, motifbreakR calls, leakage**: the first Malinois notebook;
- **P(bound | ref), P(bound | alt)**: `Malinois MPRA 5`.

New files go to `fcav_more/` and `malinois/more/`, so nothing of the 251-TF runs is touched.

| section | runtime | time |
|---|---|---|
| 1 sites for the new TFs | CPU; several sessions can split the list (`N_SESSIONS`, `PART` in 1.2) | hours (as for the pooled run) |
| 2 Malinois hits, motifbreakR calls, leakage | CPU, after all of Section 1 | ~20–40 min |
| 3 fits, P(bound \| ref), P(bound \| alt) | GPU | a few hours |
| 4 evaluation: 251 TFs vs new TFs vs all | CPU | ~10–20 min |

In [ ]:
!pip install -q pymemesuite pyfaidx
from google.colab import drive
drive.mount('/content/drive')

---
# Section 1 — sites for the new TFs *(CPU runtime)*

## 1.1 Setup (run first in every CPU session, also before Section 2)

Helpers copied unchanged from `Pooled Regions 1` (motif scan, ENCODE download, site building).

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, shutil
import numpy as np, pandas as pd

C = json.load(open(f"{WORK_DIR}/config.json"))
OLD_POOL = f"{WORK_DIR}/fcav_pool"
POOL = MORE = f"{WORK_DIR}/fcav_more"          # build() writes to f"{POOL}/sites"
MAL = f"{WORK_DIR}/malinois"
MD = f"{MAL}/more"
for d in (MORE, f"{MORE}/sites", MD):
    os.makedirs(d, exist_ok=True)
MWJ = json.load(open(f"{OLD_POOL}/mpra_windows.json"))
REGION_IVS = {int(r): (v["chrom"], int(v["start"]), int(v["end"])) for r, v in MWJ["regions"].items()}
CELLS = ["HepG2", "K562"]
N_PER_CLASS, MATCH_VARS, N_MATCH_BINS, UNBOUND_POOL = 2000, ["pwm", "gc", "cpg_oe"], 5, 10
BOUND_TOP, UNBOUND_MIN_DIST = 0.25, 1000
TOK, MOFF, WLEN = C["tok"], C["motif_offset"], C["window_len"]
pd.set_option("display.width", 250)


def overlaps_region(chrom, g_start):
    ws = g_start - MOFF
    return any(chrom == c and ws < e and ws + WLEN > s for c, s, e in REGION_IVS.values())


from pymemesuite.common import Sequence, MotifFile
from pymemesuite.fimo import FIMO
from pyfaidx import Fasta

with MotifFile(C["meme_path"]) as _mf:
    _all = list(_mf)
    BACKGROUND = _mf.background
MOTIF = {}
for m in _all:
    MOTIF.setdefault(m.name.decode(), m)           # first matrix with the name, as everywhere else
FIMO_SCAN = FIMO(both_strands=True, threshold=C["fimo_p"])
BASE_IDX = {b: i for i, b in enumerate("ACGT")}


def freq(name):
    f = np.asarray(MOTIF[name].frequencies, float)
    return f / f.sum(1, keepdims=True)


def log_odds(name, strand):
    f = freq(name) if strand == "+" else freq(name)[::-1, [3, 2, 1, 0]]
    return np.log2(np.clip(f, 1e-9, 1.0) / 0.25)


def site_score(seq, name, strand):
    lo = log_odds(name, strand)
    return float(sum(lo[j, BASE_IDX[b]] for j, b in enumerate(seq)))


def scan_many(name, seqs, batch=500):
    out = [[] for _ in seqs]
    for i0 in range(0, len(seqs), batch):
        chunk = [Sequence(s, name=str(i0 + k).encode()) for k, s in enumerate(seqs[i0:i0 + batch])]
        for h in FIMO_SCAN.score_motif(MOTIF[name], chunk, BACKGROUND).matched_elements:
            a, b = sorted((h.start, h.stop))
            out[int(h.source.name.decode())].append((a - 1, b, h.strand, h.pvalue))
    return out


LOCAL_FA = "/content/hg38.fa"
if not os.path.exists(LOCAL_FA):
    print("copying hg38 to local disk (~3 GB, one-off)...")
    shutil.copy(C["genome_fa"], LOCAL_FA)
    if os.path.exists(C["genome_fa"] + ".fai"):
        shutil.copy(C["genome_fa"] + ".fai", LOCAL_FA + ".fai")
GENOME = Fasta(LOCAL_FA, sequence_always_upper=True)
CHROMS = [c for c in [f"chr{i}" for i in range(1, 23)] + ["chrX"] if c in GENOME]
CHROM_LEN = {c: len(GENOME[c]) for c in CHROMS}


import requests

ENC, HDR = "https://www.encodeproject.org", {"accept": "application/json"}
ENC_DIR = f"{WORK_DIR}/encode"                      # shared download cache with the other notebooks
os.makedirs(ENC_DIR, exist_ok=True)
PEAK_COLS = ["chrom", "start", "end", "name", "score", "strand", "signal", "p", "q", "summit"]
PRIORITY = ["conservative IDR thresholded peaks", "optimal IDR thresholded peaks",
            "IDR thresholded peaks", "replicated peaks", "pseudoreplicated peaks"]


def enc_search(**params):
    r = requests.get(f"{ENC}/search/", params={**params, "format": "json", "limit": "all"},
                     headers=HDR, timeout=600)
    if r.status_code == 404:
        return []
    r.raise_for_status()
    return r.json()["@graph"]


def narrowpeak_files(assay_title, cell, target=None):
    q = {"type": "Experiment", "assay_title": assay_title, "status": "released", "biosample_ontology.term_name": cell}
    if target:
        q["target.label"] = target
    rows = []
    for e in enc_search(**q):
        for f in enc_search(type="File", dataset=e["@id"], file_format="bed", assembly="GRCh38", status="released"):
            if str(f.get("file_type", "")).startswith("bed narrowPeak"):
                rows.append({"experiment": e["accession"], "accession": f["accession"],
                             "output_type": f.get("output_type", ""),
                             "preferred_default": bool(f.get("preferred_default", False)), "href": f["href"]})
    return pd.DataFrame(rows)


def fetch_bed(row):
    path = f"{ENC_DIR}/{row['accession']}.bed.gz"
    if not os.path.exists(path):
        with requests.get(ENC + row["href"], stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as fh:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk)
        os.rename(path + ".part", path)
    df = pd.read_csv(path, sep="\t", header=None, compression="gzip").iloc[:, :10]
    df.columns = PEAK_COLS[:df.shape[1]]
    return df


def cell_targets(cell):
    exps = enc_search(type="Experiment", assay_title="TF ChIP-seq", status="released",
                      **{"biosample_ontology.term_name": cell}, field="target.label")
    return {e["target"]["label"].upper(): e["target"]["label"] for e in exps if "target" in e}


def pick_per_experiment(files):
    keep = []
    for _, g in files.groupby("experiment"):
        if g["preferred_default"].any():
            keep.append(g[g["preferred_default"]].iloc[0])
        else:
            rank = g["output_type"].map(lambda t: PRIORITY.index(t) if t in PRIORITY else 99)
            keep.append(g.loc[rank.idxmin()])
    return pd.DataFrame(keep)


def strict_peaks(files):
    # bound: top BOUND_TOP by signal within each experiment; exclusion: every peak file of the TF
    bound = []
    for _, r in pick_per_experiment(files).iterrows():
        d = fetch_bed(r)
        bound.append(d[d["signal"] >= d["signal"].quantile(1 - BOUND_TOP)])
    excl = pd.concat([fetch_bed(r) for _, r in files.iterrows()], ignore_index=True)
    return pd.concat(bound, ignore_index=True), excl

class Intervals:
    def __init__(self, df):
        self.d = {}
        for c, g in df.groupby("chrom"):
            o = np.argsort(g["start"].values)
            s, e = g["start"].values[o], g["end"].values[o]
            new = np.r_[True, s[1:] > np.maximum.accumulate(e)[:-1]]
            idx = np.flatnonzero(new)
            self.d[c] = (s[idx], np.maximum.reduceat(e, idx))

    def dist(self, chrom, pos):
        if chrom not in self.d:
            return np.inf
        st, en = self.d[chrom]
        i = np.searchsorted(st, pos, side="right") - 1
        d = np.inf
        if i >= 0:
            d = 0 if pos < en[i] else pos - en[i] + 1
        if i + 1 < len(st):
            d = min(d, st[i + 1] - pos)
        return d


def gc_frac(s):
    return (s.count("G") + s.count("C")) / len(s)


def cpg_oe(s):
    return s.count("CG") * len(s) / max(s.count("C") * s.count("G"), 1)


def site_record(m, cls, chrom, g_start, g_end, strand, p, signal=np.nan):
    ws = g_start - MOFF
    if ws < 0 or ws + WLEN > CHROM_LEN[chrom]:
        return None
    if overlaps_region(chrom, g_start):              # no window may overlap any MPRA region
        return None
    win = GENOME[chrom][ws:ws + WLEN].seq
    if "N" in win:
        return None
    return {"cls": cls, "chrom": chrom, "g_start": int(g_start), "strand": strand, "p": float(p), "signal": float(signal),
            "pwm": site_score(win[MOFF:MOFF + g_end - g_start], m, strand), "window": win}


def spaced(recs):
    kept, occ = [], set()
    for r in sorted(recs, key=lambda r: r["p"]):
        b = (r["chrom"], r["g_start"] // WLEN)
        if any((b[0], b[1] + k) in occ for k in (-1, 0, 1)):
            continue
        kept.append(r)
        occ.add(b)
    return kept


def match_pwm(bound, unbound, rng):
    # joint quantile bins of MATCH_VARS over the pooled sites; one bound to one unbound within each bin
    rec = bound + unbound
    if not bound or not unbound:
        return [], []
    for r in rec:
        r["gc"], r["cpg_oe"] = gc_frac(r["window"]), cpg_oe(r["window"])
    P = pd.DataFrame({v: [r[v] for r in rec] for v in MATCH_VARS})
    P["cls"] = [1] * len(bound) + [0] * len(unbound)
    keys = [pd.qcut(P[v].rank(method="first"), N_MATCH_BINS, labels=False) for v in MATCH_VARS]
    pairs = []
    for _, g in P.groupby(keys):
        b = rng.permutation(np.array(g.index[g["cls"] == 1]))
        u = rng.permutation(np.array(g.index[g["cls"] == 0]))
        k = min(len(b), len(u))
        pairs += list(zip(b[:k], u[:k]))
    pairs = [pairs[i] for i in rng.permutation(len(pairs))[:N_PER_CLASS]]
    return [rec[i] for i, _ in pairs], [rec[j] for _, j in pairs]


def spans(m):
    w = len(freq(m))
    return w, MOFF // TOK, -(-(MOFF + w) // TOK)


def build(m, files_by_cell):
    t0 = time.time()
    w = len(freq(m))
    peaks = {c: strict_peaks(f) for c, f in files_by_cell.items()}
    excl = Intervals(pd.concat([e for _, e in peaks.values()], ignore_index=True))   # every peak, every cell
    bound, unbound, n_pk = [], [], {}
    for c, (bpk, _) in peaks.items():
        pk = bpk[bpk["chrom"].isin(CHROMS)].copy()
        pk["abs_summit"] = np.where(pk["summit"] >= 0, pk["start"] + pk["summit"], (pk["start"] + pk["end"]) // 2)
        pk = pk.sort_values("signal", ascending=False).drop_duplicates(["chrom", "abs_summit"])
        pk = pk[(pk["abs_summit"] >= C["peak_half"]) & (pk["abs_summit"] + C["peak_half"] <= pk["chrom"].map(CHROM_LEN))]
        n_pk[c] = len(pk)
        chroms, summits, sig = pk["chrom"].tolist(), pk["abs_summit"].astype(int).tolist(), pk["signal"].tolist()
        seqs = [GENOME[ch][s - C["peak_half"]:s + C["peak_half"]].seq for ch, s in zip(chroms, summits)]
        for ch, s, sg, hits in zip(chroms, summits, sig, scan_many(m, seqs)):
            if not hits:
                continue
            a, b, st, p = min(hits, key=lambda h: h[3])
            if abs((a + b) / 2 - C["peak_half"]) > C["summit_dist"] or b - a != w:
                continue
            r = site_record(m, 1, ch, s - C["peak_half"] + a, s - C["peak_half"] + b, st, p, sg)
            if r:
                r["cell"] = c
                bound.append(r)
        want, got = UNBOUND_POOL * N_PER_CLASS // len(peaks), 0
        for i0 in range(0, len(DHS[c]), 2000):
            if got >= want:
                break
            chunk = DHS[c][i0:i0 + 2000]
            for (ch, s, e), hits in zip(chunk, scan_many(m, [GENOME[ch][s:e].seq for ch, s, e in chunk])):
                for a, b, st, p in hits:
                    if b - a != w or excl.dist(ch, s + (a + b) // 2) < UNBOUND_MIN_DIST:
                        continue
                    r = site_record(m, 0, ch, s + a, s + b, st, p)
                    if r:
                        r["cell"] = c
                        unbound.append(r)
                        got += 1
    bound, unbound = spaced(bound), spaced(unbound)
    rng = np.random.default_rng(C["seed"])
    nb, nu = len(bound), len(unbound)
    bound, unbound = match_pwm(bound, unbound, rng)
    rec = bound + unbound
    for r in rec:
        r["gc"], r["cpg_oe"] = gc_frac(r["window"]), cpg_oe(r["window"])
    w, K0, K1 = spans(m)
    ok = min(len(bound), len(unbound)) >= 50
    json.dump({"tf": m, "status": "built" if ok else f"too few sites (bound {nb}, unbound {nu})", "width": w,
               "K0": K0, "K1": K1, "pool_bound": nb, "pool_unbound": nu, "records": rec if ok else []},
              open(f"{POOL}/sites/{m}.json", "w"))
    by_cell = "  ".join(f"{c} {sum(r['cell'] == c for r in bound)}/{sum(r['cell'] == c for r in unbound)}" for c in CELLS)
    print(f"{m:10s} strict peaks {'+'.join(str(n_pk.get(c, 0)) for c in CELLS):>13s}  pool bound {nb:5d} unbound {nu:5d}"
          f" -> matched {len(bound):5d} per class (bound/unbound by cell: {by_cell})"
          f"{'' if ok else '  TOO FEW'}  [{time.time() - t0:.0f}s]", flush=True)


LABEL = {c: cell_targets(c) for c in CELLS}
OLD = {f[:-5] for f in os.listdir(f"{OLD_POOL}/sites") if f.endswith(".json")}
CAND = sorted(m for m in MOTIF if "::" not in m and m not in OLD and any(m.upper() in LABEL[c] for c in CELLS))
print("  ".join(f"{c}: {len(LABEL[c])} ChIP-seq targets" for c in CELLS))
print(f"{len(MOTIF)} JASPAR matrices; {len(OLD)} TFs already in fcav_pool; {len(CAND)} new TFs with HepG2/K562 ChIP-seq")

## 1.2 Build the sites

Several CPU sessions can run this at the same time: give each the same `N_SESSIONS` and a different `PART`
(0, 1, ...). Each takes every `N_SESSIONS`-th TF; a TF already in `fcav_more/sites/` is skipped, so a session can
also simply be restarted.

In [ ]:
N_SESSIONS, PART = 1, 0          # e.g. three sessions: N_SESSIONS = 3 and PART = 0 / 1 / 2

DN_FILES = {}
for c in CELLS:
    f = narrowpeak_files("DNase-seq", c)
    DN_FILES[c] = f[f["preferred_default"]] if f["preferred_default"].any() else f
DHS = {}
for c in CELLS:
    _d = pd.concat([fetch_bed(r) for _, r in DN_FILES[c].iterrows()], ignore_index=True)
    _d = _d[_d["chrom"].isin(CHROMS)].sample(frac=1, random_state=C["seed"])
    DHS[c] = list(zip(_d["chrom"], _d["start"].astype(int), _d["end"].astype(int)))
    print(f"{c} DNase peaks: {len(DHS[c]):,}")
del _d

todo = [m for i, m in enumerate(CAND) if i % N_SESSIONS == PART]
print(f"session {PART + 1} of {N_SESSIONS}: {len(todo)} TFs")
t_all = time.time()
for k, m in enumerate(todo):
    if os.path.exists(f"{MORE}/sites/{m}.json"):
        continue
    fb = {c: narrowpeak_files("TF ChIP-seq", c, LABEL[c][m.upper()]) for c in CELLS if m.upper() in LABEL[c]}
    fb = {c: f for c, f in fb.items() if len(f)}
    if not fb:
        json.dump({"tf": m, "status": "no ChIP-seq peak files", "records": []}, open(f"{MORE}/sites/{m}.json", "w"))
        continue
    print(f"[{k + 1}/{len(todo)}] ", end="")
    try:
        build(m, fb)
    except Exception as e:                       # one bad TF must not stop the session
        print(f"{m}: failed ({type(e).__name__}: {e})")
        json.dump({"tf": m, "status": f"failed: {type(e).__name__}", "records": []}, open(f"{MORE}/sites/{m}.json", "w"))
print(f"done [{(time.time() - t_all) / 60:.1f} min]")
ST = pd.Series({f[:-5]: json.load(open(f"{MORE}/sites/{f}"))["status"] for f in os.listdir(f"{MORE}/sites") if f.endswith(".json")})
print(f"\n{len(ST)} of {len(CAND)} new TFs done; status:")
print(ST.map(lambda s: s if s == "built" else s.split(" (")[0]).value_counts().to_string())

---
# Section 2 — Malinois hits, motifbreakR calls and leakage for the new TFs *(CPU, after all of Section 1)*

Run 1.1 first. The same code as Sections 1.3–1.5 of the first Malinois notebook, on the new TFs with sites; the
200 bp MPRA sequences are read from hg38 (identical to the MPRA sequence, checked there). Occurrence ids continue
after those of the 251-TF run.

In [ ]:
from pymemesuite.common import Sequence, MotifFile
from pymemesuite.fimo import FIMO
VPOS = 99
NEW = sorted(f[:-5] for f in os.listdir(f"{MORE}/sites") if f.endswith(".json")
             and json.load(open(f"{MORE}/sites/{f}"))["status"] == "built")
print(f"{len(NEW)} new TFs with sites")
V = pd.read_csv(f"{MAL}/variants.csv.gz", index_col="vi", dtype={"chrom": str}, low_memory=False)
SEQV = [None] * len(V)
row = pd.Series(np.arange(len(V)), index=V.index)
for ch, g in V.groupby("chrom38"):
    s = GENOME[ch][:].seq
    for r_, p in zip(row[g.index].values, g["pos38"].values):
        SEQV[r_] = s[p - VPOS:p - VPOS + 200]
    del s
assert all(x[VPOS] == b for x, b in zip(SEQV, V["ref"])), "hg38 base at the variant differs from the reference allele"
UID0 = int(pd.read_csv(f"{MAL}/hits.csv", usecols=["uid"])["uid"].max()) + 1

In [ ]:
from pymemesuite.common import Sequence, MotifFile
from pymemesuite.fimo import FIMO

with MotifFile(C["meme_path"]) as _mf:
    _all = list(_mf)
    BACKGROUND = _mf.background
MOTIF = {}
for m in _all:
    MOTIF.setdefault(m.name.decode(), m)           # first matrix with the name, as everywhere else
FIMO_SCAN = FIMO(both_strands=True, threshold=C["fimo_p"])
TFS = list(NEW)
TFS = [m for m in TFS if m in MOTIF]
WID = {m: np.asarray(MOTIF[m].frequencies).shape[0] for m in TFS}
MAXW = max(WID.values())
LO_, HI_ = VPOS - MAXW + 1, VPOS + MAXW
CHUNKS = []
_sub = [s[LO_:HI_] for s in SEQV]
for i0 in range(0, len(_sub), 5000):
    CHUNKS.append([Sequence(x, name=str(i0 + k).encode()) for k, x in enumerate(_sub[i0:i0 + 5000])])
del _sub
t0, rows = time.time(), []
for k, m in enumerate(TFS):
    for ch in CHUNKS:
        for h in FIMO_SCAN.score_motif(MOTIF[m], ch, BACKGROUND).matched_elements:
            a, b = sorted((h.start, h.stop))
            a, b = a - 1 + LO_, b + LO_
            if a <= VPOS < b:
                rows.append((int(h.source.name.decode()), m, a, b, h.strand, h.pvalue))
    if k % 25 == 0:
        print(f"  {k + 1}/{len(TFS)} TFs, {len(rows):,} hits [{time.time() - t0:.0f}s]", flush=True)
H = pd.DataFrame(rows, columns=["vi", "motif", "a", "b", "strand", "p"])
assert (H["b"] - H["a"] == H["motif"].map(WID)).all()
H["chrom38"] = V["chrom38"].values[H["vi"]]
H["g38"] = V["pos38"].values[H["vi"]] - VPOS + H["a"]          # hg38 start of the motif
inside = (H["g38"] - MOFF >= 0) & (H["g38"] - MOFF + WLEN <= H["chrom38"].map(CHROM_LEN))
H = H[inside].reset_index(drop=True)
H["uid"] = pd.factorize(H["chrom38"] + ":" + H["g38"].astype(str) + ":" + H["strand"] + ":" + H["motif"])[0] + UID0
print(f"{len(H):,} (variant, hit) pairs; {H['vi'].nunique():,} variants in at least one hit; "
      f"{H['uid'].nunique():,} distinct occurrences; {H['motif'].nunique()} TFs  [{(time.time() - t0) / 60:.1f} min]")

In [ ]:
import re

def _read_meme(path):
    out, name, rows_, ns = {}, None, [], 20
    for line in open(path):
        t = line.split()
        if line.startswith("MOTIF"):
            if name is not None and name not in out:
                out[name] = (np.array(rows_), ns)
            name, rows_, ns = (t[2] if len(t) > 2 else t[1]), [], 20
        elif line.startswith("letter-probability"):
            mm = re.search(r"nsites=\s*([0-9.]+)", line)
            ns = float(mm.group(1)) if mm else 20
        elif name is not None and len(t) == 4 and all(re.fullmatch(r"[0-9.eE+-]+", x) for x in t):
            rows_.append([float(x) for x in t])
    if name is not None and name not in out:
        out[name] = (np.array(rows_), ns)
    return out


MBR_THRESH, MBR_NEUT = 0.85, 0.4
MBR_MATS = {}
for name, (f, ns) in _read_meme(C["meme_path"]).items():
    f = f / f.sum(1, keepdims=True)
    pp = (f * ns + 0.25) / (ns + 1)
    S = pp * (pp.max(1) - pp.min(1))[:, None]
    S = np.vstack([S.T, np.zeros(len(S))]).T
    MBR_MATS[name] = (S, S[::-1][:, [3, 2, 1, 0, 4]], S[:, :4].min(1).sum(), S[:, :4].max(1).sum())
ENCB = np.full(256, 4, np.int8)
for i, b in enumerate("ACGT"):
    ENCB[ord(b)] = i
SEQ = ENCB[np.frombuffer("".join(SEQV).encode(), np.uint8)].reshape(len(V), 200)
REF, ALT = SEQ[:, VPOS], ENCB[np.frombuffer("".join(V["alt"]).encode(), np.uint8)]
calls = []
for m, g in H[["vi", "motif"]].drop_duplicates().groupby("motif"):
    if m not in MBR_MATS:
        continue
    S, Sr, smin, smax = MBR_MATS[m]
    w = len(S)
    st = np.arange(VPOS - w + 1, VPOS + 1)
    st = st[(st >= 0) & (st + w <= 200)]
    vi = g["vi"].values
    X = SEQ[vi][:, st[:, None] + np.arange(w)[None, :]]                # (n, windows, w)
    k = np.arange(w)[None, None, :]
    rf, rr = S[k, X].sum(2), Sr[k, X].sum(2)
    kp = (VPOS - st)[None, :]
    rb, ab = REF[vi][:, None], ALT[vi][:, None]
    af, ar = rf + S[kp, ab] - S[kp, rb], rr + Sr[kp, ab] - Sr[kp, rb]
    sr, sa = np.maximum(rf, rr).max(1), np.maximum(af, ar).max(1)
    pr, pa = (sr - smin) / (smax - smin), (sa - smin) / (smax - smin)
    diff = sa - sr
    keep = ((pr > MBR_THRESH) | (pa > MBR_THRESH)) & (np.abs(diff) >= MBR_NEUT)
    calls.append(pd.DataFrame({"vi": vi[keep], "motif": m, "alleleDiff": diff[keep]}))
CALLS = pd.concat(calls, ignore_index=True)
print(f"{len(CALLS):,} motifbreakR calls on {CALLS['vi'].nunique():,} variants, {CALLS['motif'].nunique()} TFs")

In [ ]:
H["leak"] = False
for m, g in H.groupby("motif"):
    J = json.load(open(f"{MORE}/sites/{m}.json"))
    st = pd.DataFrame([(r["chrom"], r["g_start"]) for r in J["records"]], columns=["chrom", "g"])
    for ch, gg in g.groupby("chrom38"):
        s = np.sort(st.loc[st["chrom"] == ch, "g"].values)
        if not len(s):
            continue
        x = gg["g38"].values
        i = np.searchsorted(s, x)
        near = np.minimum(np.abs(x - s[np.clip(i - 1, 0, len(s) - 1)]), np.abs(x - s[np.clip(i, 0, len(s) - 1)]))
        H.loc[gg.index, "leak"] = near < WLEN
u = H.drop_duplicates("uid")
print(f"occurrences overlapping a fitted site of the same TF: {int(u['leak'].sum()):,} of {len(u):,} "
      f"({u['leak'].mean():.1%}); they get no P(bound)")
H.to_csv(f"{MD}/hits.csv", index=False)
CALLS.to_csv(f"{MD}/calls.csv", index=False)
print(f"saved to {MD}: {len(V):,} variants, {len(H):,} hit rows, {len(CALLS):,} calls")

---
# Section 3 — fits, P(bound | ref) and P(bound | alt) for the new TFs *(GPU runtime)*

## 3.1 Setup; readout and estimator copied unchanged from `Pooled Regions 1c`

In [ ]:
!pip install -q -U "transformers==4.57.6" "tokenizers==0.22.0" "huggingface_hub==0.34.0" "safetensors==0.4.5"
!pip install -q scikit-learn scipy pandas pyfaidx
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, gc, sys, shutil
import numpy as np, pandas as pd
import torch
import torch.nn.functional as F
from contextlib import contextmanager
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from pyfaidx import Fasta
from transformers import AutoTokenizer, AutoModel, AutoConfig, AutoModelForMaskedLM, AutoModelForCausalLM

C = json.load(open(f"{WORK_DIR}/config.json"))
POOL = f"{WORK_DIR}/fcav_pool"
MORE = f"{WORK_DIR}/fcav_more"
MAL = f"{WORK_DIR}/malinois"
MD = f"{MAL}/more"
MWJ = json.load(open(f"{POOL}/mpra_windows.json"))
TAG, LAYER = "nt500m", 24
REGISTRY = {"nt500m": dict(hf="InstaDeepAI/nucleotide-transformer-v2-500m-multi-species", tok_map=("kmer", 6),
                           causal=False, batch=32)}
FIT_LAYERS = {"nt500m": [LAYER]}
BASE_WEIGHTED = False
BATCH = 32
WLEN, MOFF = C["window_len"], C["motif_offset"]
DEV = torch.device("cuda")
COMP = str.maketrans("ACGTN", "TGCAN")
rc = lambda s: s.translate(COMP)[::-1]
FITD = f"{MORE}/fits/{TAG}"
for d in (FITD, f"{MD}/p_ref", f"{MD}/palt"):
    os.makedirs(d, exist_ok=True)
LOCAL_FA = "/content/hg38.fa"
if not os.path.exists(LOCAL_FA):
    print("copying hg38 to local disk (~3 GB, one-off)...")
    shutil.copy(C["genome_fa"], LOCAL_FA)
    if os.path.exists(C["genome_fa"] + ".fai"):
        shutil.copy(C["genome_fa"] + ".fai", LOCAL_FA + ".fai")
GENOME = Fasta(LOCAL_FA, sequence_always_upper=True)
NEW = sorted(f[:-5] for f in os.listdir(f"{MORE}/sites") if f.endswith(".json")
             and json.load(open(f"{MORE}/sites/{f}"))["status"] == "built")
print(f"{len(NEW)} new TFs with sites")

In [ ]:
def load_tokenizer(tag):
    spec = REGISTRY[tag]
    tokz = AutoTokenizer.from_pretrained(spec["hf"], trust_remote_code=True)
    tokz.padding_side = "right"          # several of these default to left (HyenaDNA)
    if tokz.pad_token is None:
        tokz.pad_token = tokz.eos_token if tokz.eos_token is not None else tokz.unk_token
    return tokz


def token_bases(spec, n):
    # (n, 2) base interval of each sequence token, for the arithmetic maps
    kind = spec["tok_map"][0]
    if kind == "kmer":
        k = spec["tok_map"][1]
        return np.stack([np.arange(n) * k, (np.arange(n) + 1) * k], 1)
    if kind == "char":
        return np.stack([np.arange(n), np.arange(n) + 1], 1)
    raise RuntimeError("offsets are resolved by the tokenizer, not here")


def encode(tokz, spec, seqs):
    # input_ids (B, T), attention mask (B, T), and per sequence: the positions of its sequence tokens in input_ids
    # and the base interval each one covers
    kind = spec["tok_map"][0]
    if kind == "offsets":
        assert tokz.is_fast, f"{spec['hf']} has no fast tokenizer, so offset mapping is unavailable"
        enc = tokz(seqs, return_tensors="pt", padding=True, return_offsets_mapping=True)
    else:
        enc = tokz(seqs, return_tensors="pt", padding=True)
    ids = enc["input_ids"]
    am = enc.get("attention_mask")
    if am is None:
        am = (ids != tokz.pad_token_id).long()
    amn = am.numpy()
    pos, spans = [], []
    if kind == "offsets":
        om = enc["offset_mapping"].numpy()
        for b in range(len(seqs)):
            keep = (om[b, :, 1] > om[b, :, 0]) & (amn[b] == 1)
            pos.append(np.flatnonzero(keep))
            spans.append(om[b, keep])
    else:
        for b in range(len(seqs)):
            sp = np.array(tokz.get_special_tokens_mask(ids[b].tolist(), already_has_special_tokens=True))
            p = np.flatnonzero((sp == 0) & (amn[b] == 1))
            pos.append(p)
            spans.append(token_bases(spec, len(p)))
        assert len({len(p) for p in pos}) == 1, f"{spec['hf']}: windows of equal length gave different token counts"
    return ids, am, pos, spans


def span_weights(bases, lo, hi):
    # per-token weight for the base interval [lo, hi)
    ov = np.minimum(bases[:, 1], hi) - np.maximum(bases[:, 0], lo)
    ov = np.clip(ov, 0, None).astype(np.float32)
    if not BASE_WEIGHTED:
        ov = (ov > 0).astype(np.float32)
    assert ov.sum() > 0, f"no token covers bases [{lo}, {hi})"
    return ov / ov.sum()


def load_model(tag):
    spec = REGISTRY[tag]
    # the class the checkpoint ships its own code for (NT-v2 has AutoModelForMaskedLM but no AutoModel)
    amap = getattr(AutoConfig.from_pretrained(spec["hf"], trust_remote_code=True), "auto_map", None) or {}
    cls = next((c for k, c in (("AutoModel", AutoModel), ("AutoModelForMaskedLM", AutoModelForMaskedLM),
                               ("AutoModelForCausalLM", AutoModelForCausalLM)) if k in amap), AutoModel)
    mdl = cls.from_pretrained(spec["hf"], trust_remote_code=True).to(DEV).eval()
    # DNABERT-2's triton flash-attention kernel does not compile on current triton; its code falls back to PyTorch
    for name in {type(m).__module__ for m in mdl.modules()}:
        mod = sys.modules.get(name)
        if mod is not None and getattr(mod, "flash_attn_qkvpacked_func", None) is not None:
            mod.flash_attn_qkvpacked_func = None
            print(f"{tag}: triton flash-attention disabled in {name.split('.')[-1]}; using PyTorch attention")
    if spec.get("out") == "dnabert2" and getattr(mdl, "pooler", None) is not None:
        mdl.pooler = None
    return mdl


POS, DEPTH, LOADED = {}, {}, {}


@torch.no_grad()
def hidden_stack(tag, model, ids, am):
    # one (layers, T_j, H) float tensor per sequence; only the layers in POS[tag] once prepare() has set it
    spec = REGISTRY[tag]
    if spec.get("out") == "dnabert2":
        # DNABERT-2 returns every encoder layer unpadded; one sequence at a time keeps the token order
        out = []
        for j in range(ids.shape[0]):
            t = ids[j][am[j].bool()][None].to(DEV)
            layers, _ = model(t, attention_mask=torch.ones_like(t), output_all_encoded_layers=True)
            S = torch.stack([x.reshape(t.shape[1], -1) for x in layers], 0).float()
            out.append(S[POS[tag]] if tag in POS else S)
        return out
    kw = {} if spec["causal"] else {"attention_mask": am.to(DEV)}
    hs = model(input_ids=ids.to(DEV), output_hidden_states=True, **kw).hidden_states
    if tag in POS:
        hs = [hs[i] for i in POS[tag]]
    S = torch.stack(hs, 1).float()
    return [S[j] for j in range(S.shape[0])]


PROBE = [h["window"] for h in MWJ["hits"][:4]]


def prepare(tag):
    # load the model; from one forward pass: number of layers, token tiling, and where FIT_LAYERS sit in the stack
    spec = REGISTRY[tag]
    mdl = load_model(tag)
    tokz = LOADED.get(tag, {}).get("tokz") or load_tokenizer(tag)
    LOADED[tag] = dict(spec=spec, tokz=tokz)
    POS.pop(tag, None)
    ids, am, pos, spans = encode(tokz, spec, PROBE[:2])
    S = hidden_stack(tag, mdl, ids, am)
    for j, Sj in enumerate(S):
        assert Sj.ndim == 3 and int(pos[j].max()) < Sj.shape[1]
        b = spans[j]
        assert b[0, 0] == 0 and b[-1, 1] == WLEN and (b[1:, 0] == b[:-1, 1]).all(), f"{tag}: tokens do not tile"
    nl = S[0].shape[0]
    lids = list(range(1, nl + 1)) if spec.get("out") == "dnabert2" else list(range(nl))   # DNABERT-2: no embedding layer
    DEPTH[tag] = lids[-1]
    miss = [l for l in FIT_LAYERS[tag] if l not in lids]
    assert not miss, f"{tag}: layers {miss} do not exist (ids {lids[0]}..{lids[-1]})"
    POS[tag] = [lids.index(l) for l in FIT_LAYERS[tag]]
    print(f"{tag}: {nl} layers (depth {DEPTH[tag]}), fitting {FIT_LAYERS[tag]}, hidden {S[0].shape[2]}, "
          f"{len(pos[0])} tokens for {WLEN} bp")
    del S
    return mdl


@torch.no_grad()
def readout(tag, model, seqs, lohi):
    # (n, layers, D): D = hidden, or 2 * hidden for a causal model (forward ‖ reverse complement)
    L = LOADED[tag]
    tokz, spec = L["tokz"], L["spec"]
    passes = [(seqs, lohi)]
    if spec["causal"]:
        passes.append(([rc(s) for s in seqs], [(WLEN - b, WLEN - a) for a, b in lohi]))
    bs = spec.get("batch", BATCH)
    parts = []
    for ss, lh in passes:
        out = None
        for i in range(0, len(ss), bs):
            chunk, lhc = ss[i:i + bs], lh[i:i + bs]
            ids, am, pos, spans = encode(tokz, spec, chunk)
            S = hidden_stack(tag, model, ids, am)
            if out is None:
                out = np.zeros((len(ss), S[0].shape[0], S[0].shape[2]), np.float16)
            for j in range(len(chunk)):
                w = torch.tensor(span_weights(spans[j], *lhc[j]), device=DEV)
                p = torch.tensor(pos[j], device=DEV)
                out[i + j] = (S[j][:, p] * w[None, :, None]).sum(1).half().cpu().numpy()
            del S
        parts.append(out)
    if len(parts) == 1:
        return parts[0]
    if spec.get("rc_mode", "concat") == "mean":
        return ((parts[0].astype(np.float32) + parts[1].astype(np.float32)) / 2).astype(np.float16)
    return np.concatenate(parts, 2)

In [ ]:
torch.backends.cuda.preferred_linalg_library("cusolver")
LAMS = torch.logspace(4, -1, 11, device=DEV)      # L2 penalties tried, strongest first (standardised features, summed log-loss)
NEWTON = 15                                       # max Newton steps per penalty
INNER = 3                                         # inner CV folds for the penalty
FAST = True       # Hessian in TF32 / Cholesky in float32, float64 fallback for the layers where that fails


@contextmanager
def _tf32(on):
    old = torch.backends.cuda.matmul.allow_tf32
    torch.backends.cuda.matmul.allow_tf32 = on
    try:
        yield
    finally:
        torch.backends.cuda.matmul.allow_tf32 = old


def logit_path(X, y):
    # L2 logistic regression for every layer at once and every penalty in LAMS (warm-started, strongest first).
    # X: (layers, n, d) standardised; y: (n,) 0/1. Returns W (lams, layers, d + 1) = [weights, intercept].
    nl, n, d = X.shape
    Xa = torch.cat([X, torch.ones(nl, n, 1, device=X.device)], 2)
    pen = torch.ones(d + 1, device=X.device); pen[-1] = 0                # intercept not penalised
    w = torch.zeros(nl, d + 1, device=X.device)
    w[:, -1] = torch.logit(y.mean().clamp(1e-3, 1 - 1e-3))

    def loss(w, lam):
        z = torch.einsum("lnd,ld->ln", Xa, w)
        return F.binary_cross_entropy_with_logits(z, y.expand_as(z), reduction="none").sum(1) + lam / 2 * (pen * w * w).sum(1)

    out = []
    for lam in LAMS.tolist():
        for _ in range(NEWTON):
            z = torch.einsum("lnd,ld->ln", Xa, w)
            p = torch.sigmoid(z)
            g = torch.einsum("lnd,ln->ld", Xa, p - y) + lam * pen * w
            s_ = p * (1 - p)
            with _tf32(FAST):
                H = (Xa * s_[..., None]).transpose(1, 2) @ Xa + torch.diag(lam * pen + 1e-6)
            if FAST:
                Lc, info = torch.linalg.cholesky_ex(H)
                step = torch.cholesky_solve(g[..., None], Lc)[..., 0]
                bad = (info > 0) | ~torch.isfinite(step).all(1)
                if bad.any():                                                 # exact Hessian, float64, those layers only
                    Xb = Xa[bad].double()
                    Hb = (Xb * s_[bad].double()[..., None]).transpose(1, 2) @ Xb + torch.diag(lam * pen + 1e-6).double()
                    Lb, ib = torch.linalg.cholesky_ex(Hb)
                    sb = torch.cholesky_solve(g[bad].double()[..., None], Lb)[..., 0].float()
                    sb[ib > 0] = 0
                    step[bad] = sb
                    del Xb, Hb, Lb
            else:
                Lc, info = torch.linalg.cholesky_ex(H.double())
                step = torch.cholesky_solve(g.double()[..., None], Lc)[..., 0].float()
                step[info > 0] = 0
            step = torch.where(torch.isfinite(step), step, torch.zeros_like(step))
            ts = torch.tensor([1, .5, .25, .125, 0], device=X.device)
            L = torch.stack([loss(w - t * step, lam) for t in ts])         # (steps, layers); t = 0 keeps w
            L = torch.nan_to_num(L, nan=float("inf"))
            t = ts[L.argmin(0)][:, None]
            w = w - t * step
            if (t * step).abs().max() < 1e-4:
                break
        out.append(w.clone())
    return torch.stack(out)


def bce(Xte, yte, W):
    # held-out summed log-loss of every (penalty, layer); Xte (layers, n, d), W (lams, layers, d + 1)
    z = torch.einsum("lnd,kld->kln", Xte, W[..., :-1]) + W[..., -1:]
    return F.binary_cross_entropy_with_logits(z, yte.expand_as(z), reduction="none").sum(2)


def fit_logit(X, y, seed):
    # penalty per layer by INNER-fold CV log-loss on these sites, then the model on all of them at that penalty
    ll = torch.zeros(len(LAMS), X.shape[0], device=X.device)
    yn = y.cpu().numpy()
    for itr, ite in StratifiedKFold(INNER, shuffle=True, random_state=seed).split(np.zeros(len(yn)), yn):
        itr, ite = torch.tensor(itr, device=X.device), torch.tensor(ite, device=X.device)
        ll += bce(X[:, ite], y[ite], logit_path(X[:, itr], y[itr]))
    k = torch.nan_to_num(ll, nan=float("inf")).argmin(0)
    Wp = logit_path(X, y)
    return Wp[k, torch.arange(X.shape[0])], k.cpu().numpy(), Wp


def fit_one(X, y, seed, Hh):
    # 5 outer folds, penalty by inner CV; out-of-fold AUC per layer; each hit's logit averaged over the 5 fold models
    folds = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y))
    Xg = torch.tensor(X, dtype=torch.float32, device=DEV).transpose(0, 1).contiguous()
    yg = torch.tensor(y, dtype=torch.float32, device=DEV)
    Hg = torch.tensor(Hh, dtype=torch.float32, device=DEV).transpose(0, 1).contiguous()
    nl = Xg.shape[0]
    oof = np.zeros((len(y), nl), np.float32)
    dk, qk, kk = [], [], []
    for f, (tr, te) in enumerate(folds):
        mu, sd = Xg[:, tr].mean(1, keepdim=True), Xg[:, tr].std(1, keepdim=True) + 1e-6
        W, ki, _ = fit_logit((Xg[:, tr] - mu) / sd, yg[tr], seed + f)
        kk.append(ki)
        oof[te] = (torch.einsum("lnd,ld->ln", (Xg[:, te] - mu) / sd, W[:, :-1]) + W[:, -1:]).cpu().numpy().T
        dh = (torch.einsum("lnd,ld->ln", (Hg - mu) / sd, W[:, :-1]) + W[:, -1:]).cpu().numpy().T
        dk.append(dh); qk.append(1 / (1 + np.exp(-dh)))
    assert np.isfinite(oof).all(), "non-finite out-of-fold logits"
    auc = np.array([roc_auc_score(y, oof[:, li]) for li in range(nl)])
    mu, sd = Xg.mean(1, keepdim=True), Xg.std(1, keepdim=True) + 1e-6
    Wp = logit_path((Xg - mu) / sd, yg)
    kall = np.median(np.array(kk), 0).round().astype(int)
    W = Wp[torch.tensor(kall, device=DEV), torch.arange(nl, device=DEV)]
    res = dict(auc=auc, oof=oof, cls=y, w_all=W.cpu().numpy(), mu=mu[:, 0].cpu().numpy(), sd=sd[:, 0].cpu().numpy(),
               lam_all=LAMS.cpu().numpy()[kall], dec_hit=np.mean(dk, 0).astype(np.float32),
               prob_hit=np.mean(qk, 0).astype(np.float32))
    del Xg, Hg, Wp; gc.collect(); torch.cuda.empty_cache()
    return res

## 3.2 Fit every new TF (layer 24); one `fcav_more/fits/nt500m/<TF>.npz` each, resumable

In [ ]:
mdl = prepare(TAG)
todo = [m for m in NEW if not os.path.exists(f"{FITD}/{m}.npz")]
print(f"{len(todo)} of {len(NEW)} TFs to fit")
t_m = time.time()
for k, m in enumerate(todo):
    t0 = time.time()
    J = json.load(open(f"{MORE}/sites/{m}.json"))
    R = J["records"]
    w = int(J["width"])
    y = np.array([r["cls"] for r in R])
    X = readout(TAG, mdl, [r["window"] for r in R], [(MOFF, MOFF + w)] * len(R))
    assert np.isfinite(X).all(), f"{m}: a hidden state overflowed float16"
    res = fit_one(X, y, C["seed"], X[:2])                 # no Kircher hits for these TFs: a dummy pair, discarded
    tmp = f"{FITD}/{m}.part.npz"
    np.savez(tmp, layers=np.array([LAYER]), n_layers=DEPTH[TAG], model=TAG, base_weighted=BASE_WEIGHTED,
             **{key: res[key] for key in ("auc", "oof", "cls", "w_all", "mu", "sd", "lam_all")})
    os.replace(tmp, f"{FITD}/{m}.npz")
    left = (time.time() - t_m) / (k + 1) * (len(todo) - k - 1) / 3600
    print(f"[{k + 1}/{len(todo)}] {m:10s} sites {len(y):5d}  AUC L{LAYER} {res['auc'][0]:.3f}  "
          f"[{time.time() - t0:.0f}s; ~{left:.1f} h left]", flush=True)
    del X; gc.collect(); torch.cuda.empty_cache()

## 3.3 P(bound | ref) of every new occurrence and P(bound | alt) of every (variant, occurrence) pair

In [ ]:
def direction(m):
    z = np.load(f"{FITD}/{m}.npz", allow_pickle=False)
    li = [int(l) for l in z["layers"]].index(LAYER)
    return z["w_all"][li].astype(np.float64), z["mu"][li].astype(np.float64), z["sd"][li].astype(np.float64)


def decision(m, windows, w_motif, mdl):
    w, mu, sd = direction(m)
    X = readout(TAG, mdl, windows, [(MOFF, MOFF + w_motif)] * len(windows))[:, 0].astype(np.float64)
    return ((X - mu) / sd) @ w[:-1] + w[-1]


V = pd.read_csv(f"{MAL}/variants.csv.gz", index_col="vi", dtype={"chrom": str}, low_memory=False,
                usecols=["vi", "chrom38", "pos38", "ref", "alt"])


def windows_of(g, alt):
    out = []
    for c, s, vi in zip(g["chrom38"].values, g["g38"].values, g["vi"].values):
        ws = int(s) - MOFF
        x = GENOME[c][ws:ws + WLEN].seq
        if alt:
            k = int(V.at[vi, "pos38"]) - ws
            assert 0 <= k < WLEN and x[k] == V.at[vi, "ref"], f"variant {vi}: reference base not at {k}"
            x = x[:k] + V.at[vi, "alt"] + x[k + 1:]
        assert len(x) == WLEN and "N" not in x
        out.append(x)
    return out


H2 = pd.read_csv(f"{MD}/hits.csv")
H2 = H2[~H2["leak"] & H2["motif"].isin([f[:-4] for f in os.listdir(FITD) if f.endswith(".npz") and not f.endswith(".part.npz")])]
t_all = time.time()
for kind, out_dir, keys in (("ref", f"{MD}/p_ref", ["uid"]), ("alt", f"{MD}/palt", ["vi", "uid"])):
    G = H2.drop_duplicates(keys)
    todo = [(m, g) for m, g in G.groupby("motif") if not os.path.exists(f"{out_dir}/{m}.csv")]
    print(f"P(bound | {kind}): {len(todo)} TFs, {sum(len(g) for _, g in todo):,} windows to do", flush=True)
    for k, (m, g) in enumerate(todo):
        d = decision(m, windows_of(g, kind == "alt"), int(g["b"].iloc[0] - g["a"].iloc[0]), mdl)
        df = pd.DataFrame({"uid": g["uid"].values, "P": 1 / (1 + np.exp(-d))}) if kind == "ref" else \
             pd.DataFrame({"vi": g["vi"].values, "uid": g["uid"].values, "dec_alt": d})
        df.to_csv(f"{out_dir}/{m}.part.csv", index=False)
        os.replace(f"{out_dir}/{m}.part.csv", f"{out_dir}/{m}.csv")
        if k % 25 == 0:
            print(f"  [{k + 1}/{len(todo)}] {m}  [{(time.time() - t_all) / 60:.1f} min]", flush=True)
print(f"done [{(time.time() - t_all) / 60:.1f} min]")
del mdl; gc.collect(); torch.cuda.empty_cache()

In [ ]:
from google.colab import runtime
runtime.unassign()

---
# Section 4 — evaluation *(CPU runtime)*

Three TF sets, each split by the held-out ChIP AUC of the TF's CAV (all, ≥ 0.70, ≥ 0.75, ≥ 0.80):
**the 251 TFs** of the earlier runs (should reproduce `Malinois MPRA 5`), **the new TFs only**, and **all TFs**.
Per set, over the variants with at least one occurrence of those TFs: Pearson of ΣΔP with Δ, of summed motifbreakR,
and the out-of-sample gain of summed motifbreakR + ΣΔP over summed motifbreakR (chromosome halves; p N2 = share of
100 shuffles of ΔP among the pairs of the same TF with at least this gain).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, time
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

MAL = f"{WORK_DIR}/malinois"
MD = f"{MAL}/more"
OUT = f"{MD}/eval"
os.makedirs(OUT, exist_ok=True)
CELLS3 = ["HepG2", "K562", "SKNSH"]
TIERS = [("all", 0.0), ("AUC >= 0.70", 0.70), ("AUC >= 0.75", 0.75), ("AUC >= 0.80", 0.80)]
ACTIVE_MIN, Z_CLEAR, N_PERM, LAYER = 1.0, 2.0, 100, 24
pd.set_option("display.width", 250)
fmt = lambda x: f"{x:+.3f}"

V = pd.read_csv(f"{MAL}/variants.csv.gz", index_col="vi", dtype={"chrom": str}, low_memory=False)
rd = lambda d: pd.concat([pd.read_csv(f"{d}/{f}") for f in os.listdir(d) if f.endswith(".csv") and not f.endswith(".part.csv")])


def pairs(hits, p_dir, palt_dir, fit_dir, calls):
    H = pd.read_csv(hits)
    H = H[~H["leak"]].drop_duplicates(["vi", "uid"])[["vi", "uid", "motif"]]
    H = H.merge(rd(palt_dir), on=["vi", "uid"])
    H["P_ref"] = rd(p_dir).set_index("uid")["P"].reindex(H["uid"]).values
    H = H[np.isfinite(H["P_ref"])]
    H["dP"] = 1 / (1 + np.exp(-H["dec_alt"])) - H["P_ref"]
    auc = {}
    for m in H["motif"].unique():
        z = np.load(f"{fit_dir}/{m}.npz", allow_pickle=False)
        auc[m] = float(np.asarray(z["auc"])[[int(l) for l in z["layers"]].index(LAYER)])
    H["auc"] = H["motif"].map(auc)
    H = H.merge(calls[["vi", "motif"]].drop_duplicates().assign(called=True), on=["vi", "motif"], how="left")
    H["called"] = H["called"].astype("boolean").fillna(False).astype(bool)
    return H


C_OLD, C_NEW = pd.read_csv(f"{MAL}/calls.csv"), pd.read_csv(f"{MD}/calls.csv")
OLDP = pairs(f"{MAL}/hits.csv", f"{MAL}/p_nt500m", f"{MAL}/palt_nt500m", f"{WORK_DIR}/fcav_pool/fits/nt500m", C_OLD)
NEWP = pairs(f"{MD}/hits.csv", f"{MD}/p_ref", f"{MD}/palt", f"{WORK_DIR}/fcav_more/fits/nt500m", C_NEW)
CALLS = pd.concat([C_OLD, C_NEW], ignore_index=True)
ALLP = pd.concat([OLDP, NEWP], ignore_index=True)
OLD_TFS, NEW_TFS = set(OLDP["motif"]), set(NEWP["motif"])
print(f"251-TF run: {len(OLD_TFS)} TFs, {len(OLDP):,} pairs, {OLDP['vi'].nunique():,} variants")
print(f"new TFs:    {len(NEW_TFS)} TFs, {len(NEWP):,} pairs, {NEWP['vi'].nunique():,} variants "
      f"({len(set(NEWP['vi']) - set(OLDP['vi'])):,} not reached by the 251 TFs)")
print(f"all:        {len(OLD_TFS | NEW_TFS)} TFs, {ALLP['vi'].nunique():,} variants")
A = pd.Series({**OLDP.groupby("motif")["auc"].first(), **NEWP.groupby("motif")["auc"].first()})
print("\nheld-out ChIP AUC of the CAVs:")
print(pd.DataFrame({"251 TFs": A[list(OLD_TFS)].describe(), "new TFs": A[list(NEW_TFS)].describe()}).to_string(float_format=lambda x: f"{x:.3f}"))

In [ ]:
def pear(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a - a.mean(), b - b.mean()
    den = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / den) if den > 0 else np.nan


def context(L):
    L = L.sort_values(["vi", "motif"]).reset_index(drop=True)
    VI = np.unique(L["vi"].values)
    X = dict(VI=VI, cv=np.searchsorted(VI, L["vi"].values), cm=pd.factorize(L["motif"])[0], dP=L["dP"].values)
    X["base"] = np.argsort(X["cm"], kind="stable")
    Cc = CALLS[CALLS["motif"].isin(set(L["motif"])) & CALLS["vi"].isin(VI)]
    X["SUM_AD"] = np.bincount(np.searchsorted(VI, Cc["vi"].values), Cc["alleleDiff"].values, minlength=len(VI))
    chrom = V.loc[VI, "chrom"].astype(str).values
    X["FOLD"] = np.array([int(ch) % 2 if ch.isdigit() else 0 for ch in chrom])
    X["Y"] = {c: V.loc[VI, f"d_{c}"].values for c in CELLS3}
    X["SETS"] = {(c, s): np.isfinite(X["Y"][c]) & ((V.loc[VI, f"act_{c}"].values >= ACTIVE_MIN) if s == "active" else True)
                 for c in CELLS3 for s in ("all", "active")}
    return X


def cv_pear(cols, y, f):
    Z = np.column_stack([np.ones(len(y))] + cols)
    out = np.zeros(len(y))
    for k in (0, 1):
        tr, te = f != k, f == k
        out[te] = Z[te] @ np.linalg.lstsq(Z[tr], y[tr], rcond=None)[0]
    return pear(out, y)


def shuffle_tf(X, rng):
    o = np.lexsort((rng.random(len(X["dP"])), X["cm"]))
    out = np.empty_like(X["dP"]); out[X["base"]] = X["dP"][o]
    return out


def stats_(X, dP):
    sP = np.bincount(X["cv"], dP, minlength=len(X["VI"]))
    out = {}
    for key, m in X["SETS"].items():
        y, f, b = X["Y"][key[0]][m], X["FOLD"][m], X["SUM_AD"][m]
        r0 = cv_pear([b], y, f)
        out[key] = (pear(sP[m], y), r0, cv_pear([b, sP[m]], y, f) - r0)
    return out


ROWS = []
t0 = time.time()
for set_name, tfs in (("251 TFs", OLD_TFS), ("new TFs only", NEW_TFS), ("all TFs", OLD_TFS | NEW_TFS)):
    for tier, thr in TIERS:
        L = ALLP[ALLP["motif"].isin(tfs) & (ALLP["auc"] >= thr)]
        if L["vi"].nunique() < 1000:
            continue
        X = context(L)
        rng = np.random.default_rng(7)
        obs = stats_(X, X["dP"])
        null = [stats_(X, shuffle_tf(X, rng)) for _ in range(N_PERM)]
        for (c, s), (rp, r0, g) in obs.items():
            nl = np.array([n_[c, s] for n_ in null])
            ROWS.append({"TF set": set_name, "AUC tier": tier, "cell": c, "variants": s, "TFs": int(L["motif"].nunique()),
                         "n": int(X["SETS"][c, s].sum()), "Pearson ΣΔP": rp, "p N2 ΣΔP": np.mean(nl[:, 0] >= rp),
                         "Pearson summed motifbreakR (CV)": r0, "gain + ΣΔP": g, "p N2 gain": np.mean(nl[:, 2] >= g)})
        print(f"{set_name}, {tier}: {L['motif'].nunique()} TFs, {len(X['VI']):,} variants  [{(time.time() - t0) / 60:.1f} min]", flush=True)
R = pd.DataFrame(ROWS).set_index(["TF set", "AUC tier", "cell", "variants"])
R.to_csv(f"{OUT}/more_tfs.csv")
print()
print(R.to_string(float_format=fmt))

## End

New sites and fits in `fcav_more/`; Malinois hits / calls / P(bound | ref) / P(bound | alt) of the new TFs in
`malinois/more/`; the table in `malinois/more/eval/more_tfs.csv`.